# 01 · Data prep at scale: micro-batching, Artifacts, and recover/fork

**Review Radar**: turn raw product reviews into a
clean, labeled training set. These are consumed in the later chapters as input.

You'll build it up piece by piece:

1. **Micro-batching**: split the reviews into batches labeled by a warm pool of workers.
2. **`@flyte.trace`**: make the submit/poll calls to the labeling service durable checkpoints.
3. **Artifacts**: publish the validated dataset as a named, versioned `review-dataset`.
4. **recover / fork**: a strict quality gate fails the run *after* the expensive labeling
   work; you rescue it without redoing that work.

> Run this notebook **remotely**: every `flyte.run` executes on the cluster.

> **Note on types:** these tasks pass plain `list[str]` / `File` values. Task inputs/outputs
> in a notebook must be built-in types (a class defined in a cell isn't importable on the
> cluster). The matching script `workflows/01_data_prep.py` uses `@dataclass` for the same
> flow, nicer for real code, since a module *is* importable.

In [11]:
from pathlib import Path
import asyncio

import flyte
import pandas as pd
from flyte.io import File

# connect to the cluster (endpoint/project come from .flyte/config.yaml)
flyte.init_from_config(Path('.flyte') / 'config.yaml')

# one container image, shared by every task below
image = flyte.Image.from_debian_base().with_pip_packages('pandas', 'pyarrow', 'unionai-reuse>=0.1.9')

## Step 1 · A pool of reusable labeling workers

The workers share warm replicas (`ReusePolicy`), so batches skip pod startup. The driver
orchestrates them, so it declares `depends_on=[worker_env]`.

In [ ]:
worker_env = flyte.TaskEnvironment(
    name='prep_worker',
    image=image,
    resources=flyte.Resources(cpu=1, memory='512Mi'),
    # warm pool: batches reuse replicas instead of paying pod startup
    reusable=flyte.ReusePolicy(replicas=(1, 2), concurrency=8, idle_ttl=60),
)

driver_env = flyte.TaskEnvironment(
    name='prep_driver',
    image=image,
    resources=flyte.Resources(cpu=1, memory='512Mi'),
    # the driver calls the workers, so it must declare depends_on
    depends_on=[worker_env],
)

## Step 2 · Durable submit/poll with `@flyte.trace`

The reviews are synthetic (with ground-truth labels) so the demo is deterministic. The
'labeling service' is stubbed but keeps a real **submit → poll** shape. Both calls are
`@flyte.trace` (no parentheses): their inputs/outputs are persisted as spans, so a retried
batch resumes from the checkpoint instead of re-submitting.

In [ ]:
_POS = ['love it', 'works great', 'excellent quality', 'highly recommend', 'fast and reliable']
_NEG = ['broke immediately', 'terrible support', 'waste of money', 'very disappointed', 'stopped working']
_AMBIG = ['it is okay', 'does the job', 'not sure yet', 'average at best', 'mixed feelings']

# deterministic synthetic reviews, so recover/fork behave predictably
def _synth_reviews(n: int) -> tuple[list[str], list[str]]:
    texts, sentiments = [], []
    for i in range(n):
        bucket = i % 5
        if bucket in (0, 1):
            texts.append(_POS[i % len(_POS)]); sentiments.append('positive')
        elif bucket in (2, 3):
            texts.append(_NEG[i % len(_NEG)]); sentiments.append('negative')
        else:
            texts.append(_AMBIG[i % len(_AMBIG)]); sentiments.append('positive' if i % 2 else 'negative')
    return texts, sentiments

In [ ]:
# durable checkpoint: a retried batch resumes here instead of re-submitting
@flyte.trace
async def submit_labeling(texts: list[str]) -> str:
    await asyncio.sleep(0)  # stands in for a network round-trip
    return f'job-{abs(hash(tuple(texts))) % 10_000}'

@flyte.trace
async def poll_labeling(job_id: str, texts: list[str]) -> list[str]:
    predicted = []
    for t in texts:
        low = t.lower()
        if any(k in low for k in ('love', 'great', 'excellent', 'recommend', 'reliable')):
            predicted.append('positive')
        elif any(k in low for k in ('broke', 'terrible', 'waste', 'disappointed', 'stopped')):
            predicted.append('negative')
        else:
            predicted.append('positive')  # ambiguous -> best-effort guess
    return predicted

## Step 3 · The batch worker

One `process_batch` call per micro-batch → one action each, so recover/fork can reuse them
individually. It returns the predicted labels for its slice.

In [ ]:
# one micro-batch per call = one action (recover can reuse them individually)
@worker_env.task
async def process_batch(texts: list[str], batch_id: int) -> list[str]:
    job_id = await submit_labeling(texts)
    predicted = await poll_labeling(job_id, texts)
    print(f'batch {batch_id}: labeled {len(texts)} reviews (job {job_id})')
    return predicted

## Step 4 · The quality gate, and publish the Artifact

`validate_dataset` measures labeling quality (fraction matching ground truth). Below the
threshold it **fails the run**; at or above it publishes the dataset as the `review-dataset`
**Artifact** (`produces_artifacts=True` + `flyte.artifacts.new`).

In [ ]:
@driver_env.task(produces_artifacts=True)
async def validate_dataset(dataset: File, min_quality: float) -> File:
    local = await dataset.download()
    df = pd.read_parquet(local)
    quality = float((df['predicted'] == df['sentiment']).mean())
    print(f'dataset quality = {quality:.3f} (threshold {min_quality})')
    # the quality gate: below the threshold the whole run fails here
    if quality < min_quality:
        raise ValueError(f'dataset quality {quality:.3f} below threshold {min_quality}')
    # publish the validated dataset as a named, versioned Artifact
    return flyte.artifacts.new(
        dataset,
        flyte.artifacts.Metadata(
            name='review-dataset',
            description='Labeled product-review training set',
            attrs={'rows': str(len(df)), 'quality': f'{quality:.3f}'},
        ),
    )

## Step 5 · The driver: fan out, assemble, gate

The driver labels every batch in parallel, assembles the parquet dataset, then calls the
gate. Splitting `process_batch` (workers) from `validate_dataset` (gate) is what lets recover
reuse the batches while re-running only the gate.

In [ ]:
@driver_env.task
async def main(n_reviews: int = 200, batch_size: int = 25, min_quality: float = 0.95) -> File:
    texts, sentiments = _synth_reviews(n_reviews)
    batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    # fan out every batch in parallel
    with flyte.group('label-batches'):
        preds = await asyncio.gather(*(process_batch(b, i) for i, b in enumerate(batches)))
    predicted = [p for batch in preds for p in batch]
    df = pd.DataFrame({'text': texts, 'sentiment': sentiments, 'predicted': predicted})
    df.to_parquet('/tmp/review_dataset.parquet', index=False)
    dataset = await File.from_local('/tmp/review_dataset.parquet')
    return await validate_dataset(dataset, min_quality)

## Step 6 · The quality gate fails a run: now rescue it

Run with the **default** strict threshold (0.95). The synthetic data can't clear it, so the
run fails at `validate_dataset`, *after* all the batch-labeling actions succeeded. That's
exactly the situation recover/fork are for: don't throw away the expensive work.

In [ ]:
fail_run = flyte.run(main, min_quality=0.95)  # min_quality defaults to 0.95 -> fails at the gate
print('Failing run:', fail_run.name, fail_run.url)
try:
    fail_run.wait()
except Exception as e:
    print('run failed as expected:', type(e).__name__)
print('phase:', fail_run.phase)

### Recover: reuse the succeeded actions, re-run only the gate, produce an output Artifact

`flyte.rerun(..., recover=True)` reuses every succeeded action and re-runs only what failed
or never ran. We also lower `min_quality`: only the gate reads it, so the batch actions stay
reused. (Equivalent CLI: `flyte rerun <run> --recover --min_quality 0.7`.)

In [ ]:
# recover: reuse the succeeded batches, re-run only the failed gate
rec = flyte.rerun(fail_run.name, recover=True, min_quality=0.7)
print('Recover run:', rec.name, rec.url)
rec.wait()
print('phase:', rec.phase)

In [ ]:
# Confirm the batches were REUSED (RECOVERED), only the gate re-ran.
import flyte.remote
# the batches show RECOVERED; only validate_dataset re-ran
for a in flyte.remote.Action.listall(for_run_name=rec.name):
    print(f'{a.task_name:32} {a.phase}')

### Fork: same idea, but with your **current working-tree code**

`recover` replays the *source run's* code. When you've **edited** the code to fix a bug,
use **fork** instead: it rebuilds a bundle from your working tree, so changed actions
re-execute while unchanged succeeded ones are reused. Fork runs against a script file, so
from the `workflows/` folder, against a failing run of `01_data_prep.py`:

```bash
flyte fork <failing-run-name> 01_data_prep.py main --min_quality 0.7
```

| command | code source | reuse succeeded? |
|---------|-------------|------------------|
| `flyte rerun <run>` | source run | no, everything re-runs |
| `flyte rerun <run> --recover` | source run | yes |
| `flyte fork <run> file.py main` | your working tree | yes |

## Recap

- Micro-batching + reusable workers labeled the corpus in parallel, each batch its own action.
- `@flyte.trace` made the submit/poll calls durable checkpoints.
- The validated dataset is a versioned **`review-dataset` Artifact**, the hand-off to Ch. 2.
- **recover** and **fork** both rescued a failed run by reusing the expensive succeeded work;
  recover uses the source run's code, fork uses your current code.

Next: **`02_training.ipynb`** - HPO fan-out and a published model Artifact.